# Native SDK Telemetry
Inspect what each SDK natively reports back to its caller (token usage, costs, latency, model info, etc.).

In [1]:
import os
import json

from dotenv import load_dotenv

load_dotenv()
model_name = os.getenv("MODEL_NAME") or ""
message = os.getenv("MESSAGE") or ""

## Anthropic SDK

In [2]:
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model=model_name,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": message}
    ]
)

print("type:", type(response).__name__)
print("\nTop-level fields:")
for field in response.model_fields:
    print(f"  {field}: {getattr(response, field)!r}")

type: Message

Top-level fields:
  id: 'msg_01B3EoyZ7snW6i9pJZ7xZZgY'
  container: None
  content: [TextBlock(citations=None, text='## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Identified\n\n### 🔴 Critical Issues\n\n#### 1. **Missing Authentication Check**\nThe code checks *authorization* (admin role) but shows no evidence of prior *authentication*.\n```javascript\n// PROBLEM: Who verified this user is who they claim to be?\nif (user.is_admin) { // ← trusting unverified data\n    session.open();\n}\n```\n\n#### 2. **Client-Side / Untrusted Data Trust**\n`user.is_admin` may originate from:\n- A cookie\n- A JWT payload (unverified)\n- User-supplied request data\n\n```javascript\n// Attacker can simply send:\n{ "user": { "is_admin": true } }\n```\n\n#### 3. **No Input Validation / Null Checks**\n```javascript\n// These cases are unhandled:\nuser = null;           // NullPointerExcepti

/var/folders/8j/f_9x77cd28703_l8b6tlflcw0000gn/T/ipykernel_46130/3793456385.py:15: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for field in response.model_fields:


## Claude Agent SDK

In [3]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage, TextBlock

responses = []
async for msg in query(
    prompt=message,
    options=ClaudeAgentOptions(model=model_name, max_turns=1)
):
    responses.append(msg)

for msg in responses:
    print(f"type: {type(msg).__name__}")
    if hasattr(msg, '__dataclass_fields__'):
        for field in msg.__dataclass_fields__:
            print(f"  {field}: {getattr(msg, field)!r}")
    else:
        print(f"  value: {msg!r}")
    print()

type: SystemMessage
  subtype: 'init'
  data: {'type': 'system', 'subtype': 'init', 'cwd': '/Users/svespie/Code/agent-telemetry-experiment/src', 'session_id': '8097b63c-bb66-47a3-bf68-0a902c835fda', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'NotebookEdit', 'Read', 'RemoteTrigger', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'ToolSearch', 'WebFetch', 'WebSearch', 'Write'], 'mcp_servers': [{'name': 'claude.ai Google Calendar', 'status': 'needs-auth'}, {'name': 'claude.ai Gmail', 'status': 'needs-auth'}], 'model': 'claude-sonnet-4-6', 'permissionMode': 'default', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'pr-comments', 'release-notes', 'review', 'security-review', 'insights'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.87', 'o

## LangChain

In [4]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model=model_name)
response = llm.invoke(message)

print("type:", type(response).__name__)
print("\nTop-level fields:")
for key, value in response.__dict__.items():
    print(f"  {key}: {value!r}")
print("\nresponse_metadata:")
print(json.dumps(response.response_metadata, indent=2))
print("\nusage_metadata:")
print(response.usage_metadata)

type: AIMessage

Top-level fields:
  content: "## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Identified\n\n### 🔴 Critical Issues\n\n#### 1. **Missing Authentication Check**\nThe code checks **authorization** (role) but not **authentication** (identity).\n```javascript\n// VULNERABLE: No verification the user is who they claim to be\nif (user.is_admin) { session.open(); }\n\n// SAFER: Verify identity first\nif (user.isAuthenticated() && user.is_admin) { session.open(); }\n```\n\n#### 2. **Client-Side Trust / Object Tampering**\nIf `user` object originates from client input, `is_admin` can be manipulated.\n```javascript\n// ATTACK EXAMPLE:\nuser = { is_admin: true }  // Attacker crafts this object\n// Privilege escalation achieved trivially\n```\n\n#### 3. **Boolean Flag Privilege Check**\nSingle boolean flags are fragile and easily forged.\n```javascript\n// VULNERABLE\nuser.is_admin

## Pydantic AI

In [5]:
from pydantic_ai import Agent as PydanticAgent

agent = PydanticAgent(f"anthropic:{model_name}")
result = await agent.run(message)

print("type:", type(result).__name__)
print("\nTop-level attributes:")
for attr in dir(result):
    if attr.startswith("_"):
        continue
    val = getattr(result, attr)
    if callable(val):
        try:
            val = val()
            print(f"  {attr}(): {val!r}")
        except Exception:
            pass
    else:
        print(f"  {attr}: {val!r}")

type: AgentRunResult

Top-level attributes:
  all_messages(): [ModelRequest(parts=[UserPromptPart(content='Scan the following logic for vulnerabilities: if (user.is_admin) { session.open(); }', timestamp=datetime.datetime(2026, 3, 31, 1, 12, 18, 913870, tzinfo=datetime.timezone.utc))]), ModelResponse(parts=[TextPart(content='## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Identified\n\n### 🔴 Critical Issues\n\n#### 1. **Missing Authentication Check**\nThe code checks *authorization* (role) but not *authentication* (identity).\n```javascript\n// PROBLEM: No verification that user is actually logged in\nif (user.is_admin) { // user could be anonymous/unauthenticated\n    session.open();\n}\n\n// SAFER:\nif (user.is_authenticated && user.is_admin) {\n    session.open();\n}\n```\n\n#### 2. **Client-Side Trust / No Server-Side Validation**\n`is_admin` may be a client-controlled value, maki

/var/folders/8j/f_9x77cd28703_l8b6tlflcw0000gn/T/ipykernel_46130/3862523075.py:11: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  val = getattr(result, attr)


## CrewAI

In [6]:
from crewai import Agent as CrewAgent, Task, Crew, LLM

llm = LLM(model=model_name)

agent = CrewAgent(
    role="Assistant",
    goal="Respond to the user's message",
    backstory="You are a helpful assistant.",
    llm=llm
)

task = Task(
    description=message,
    expected_output="A response to the user's message",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task])
result = crew.kickoff()

print("type:", type(result).__name__)
print("\nTop-level attributes:")
for attr in dir(result):
    if not attr.startswith('_'):
        try:
            val = getattr(result, attr)
            if not callable(val):
                print(f"  {attr}: {val!r}")
        except Exception:
            pass

type: CrewOutput

Top-level attributes:
  json_dict: None
  model_computed_fields: {}
  model_config: {}
  model_extra: None
  model_fields: {'raw': FieldInfo(annotation=str, required=False, default='', description='Raw output of crew'), 'pydantic': FieldInfo(annotation=Union[BaseModel, NoneType], required=False, default=None, description='Pydantic output of Crew'), 'json_dict': FieldInfo(annotation=Union[Dict[str, Any], NoneType], required=False, default=None, description='JSON dict output of Crew'), 'tasks_output': FieldInfo(annotation=list[TaskOutput], required=False, default=[], description='Output of each task'), 'token_usage': FieldInfo(annotation=UsageMetrics, required=False, default={}, description='Processed token summary')}
  model_fields_set: {'tasks_output', 'json_dict', 'pydantic', 'token_usage', 'raw'}
  pydantic: None
  raw: '## Security Vulnerability Analysis\n\n### Code Under Review:\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## 🔴 Ident

/var/folders/8j/f_9x77cd28703_l8b6tlflcw0000gn/T/ipykernel_46130/4168727385.py:26: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  val = getattr(result, attr)
/var/folders/8j/f_9x77cd28703_l8b6tlflcw0000gn/T/ipykernel_46130/4168727385.py:26: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  val = getattr(result, attr)
